## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or (for Codespaces / remote environments)

```bash
az login --use-device-code
```

# 🍗 Rotisserie Chicken Agent with Azure AI Foundry

This notebook demonstrates building an AI agent using `Agent` and `FoundryChatClient` from `agent-framework-core`. You'll build a rotisserie chicken planning assistant for Zava Groceries stores.

## Features Covered:
- Setting up `FoundryChatClient` to connect to Azure AI Foundry
- Creating agents with custom instructions using `Agent`
- Adding function tools with the `@tool` decorator
- Configuring approval modes for secure tool execution
- Running agent queries and handling responses

### ⚠️ Important Note ⚠️
> **The `agent-framework-core` library provides a simple API for building AI agents. Use the `@tool` decorator to define function tools, and configure `approval_mode` for security.**

## Prerequisites

Before running this notebook, ensure you have:

| Requirement | Description |
|-------------|-------------|
| **Azure AI Project** | Access to a Microsoft Foundry project with deployed models |
| **Authentication** | Azure CLI installed and authenticated (`az login --use-device-code`) |
| **Environment Variables** | Set up your `.env` file (see below) |
| **Dependencies** | Required agent-framework packages installed |

### Required Environment Variables

Create a `.env` file in the project root with:
```
AI_FOUNDRY_PROJECT_ENDPOINT=https://your-project.services.ai.azure.com/api/projects/your-project-id
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o
```

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

## Step 0 — Install Required Packages

Install the `agent-framework-core` package and dependencies.

In [8]:
%pip install agent-framework-core==1.0.1 agent-framework-foundry azure-identity python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


## Step 1 — Import Libraries

Import the required libraries using the `Agent` and `FoundryChatClient` API from `agent-framework-core`:

In [9]:
import os
from pathlib import Path

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

# Load environment variables from project root
notebook_path = Path().absolute()
load_dotenv('../.env')  # Adjust path as needed

# Verify environment setup
endpoint = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')
model = os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')

print("🔧 Environment Configuration:")
print(f"✅ Project Endpoint: {endpoint[:50]}..." if endpoint else "❌ AI_FOUNDRY_PROJECT_ENDPOINT not set")
print(f"✅ Model Deployment: {model}" if model else "❌ AZURE_AI_MODEL_DEPLOYMENT_NAME not set")

🔧 Environment Configuration:
✅ Project Endpoint: https://demoagenticfoundry.services.ai.azure.com/a...
✅ Model Deployment: gpt-4.1


## Step 2 — Create the FoundryChatClient 🔌

The `FoundryChatClient` connects to your Azure AI Foundry project and provides access to deployed models.

| Parameter | Description |
|-----------|-------------|
| **project_endpoint** | Your Foundry project endpoint URL |
| **model** | The deployed model name (e.g., `gpt-4o`) |
| **credential** | Azure credential for authentication |

In [10]:
# Create the FoundryChatClient
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)

print(f"✅ Created FoundryChatClient")
print(f"🔌 Connected to: {endpoint[:50]}...")
print(f"🤖 Model: {model}")

✅ Created FoundryChatClient
🔌 Connected to: https://demoagenticfoundry.services.ai.azure.com/a...
🤖 Model: gpt-4.1


## Step 3 — Create Rotisserie Chicken Agent 🍗

Create an agent with the rotisserie chicken planning instructions. The agent will help store managers analyze chicken sales data.

**Key Concepts:**
- `Agent` wraps the chat client with instructions and tools
- `@tool` decorator defines local function tools
- `approval_mode` controls user confirmation for tool execution

### Approval Modes

| Mode | Description | Use Case |
|------|-------------|----------|
| `"never_require"` | Tool calls execute automatically | Read-only queries, development |
| `"always_require"` | Every tool call requires explicit user approval | Write operations, production |

In [11]:
# Agent system instructions
ROTISSERIE_AGENT_INSTRUCTIONS = """You are a rotisserie chicken planning assistant for Zava Groceries stores.

Your role is to help store managers:
1. Query daily financial data (chickens bought, sold, revenue, profit)
2. Query hourly sales data (chickens cooked vs sold per hour)
3. Identify trends and patterns in sales data
4. Recommend optimal cooking schedules to minimize waste
5. Track store performance across different time periods

When analyzing data:
- Focus on sell-through rates (chickens sold / chickens cooked)
- Identify peak hours (typically lunch 11am-1pm and dinner 5pm-7pm)
- Flag high-waste periods where cooked > sold
- Calculate profit margins when financial data is available

Always explain your findings in clear, simple language that store associates can understand.
"""

# Create the Rotisserie Chicken Agent
rotisserie_agent = Agent(
    client=client,
    name="RotisserieChickenAgent",
    instructions=ROTISSERIE_AGENT_INSTRUCTIONS,
)

print(f"✅ Created agent: {rotisserie_agent.name}")

✅ Created agent: RotisserieChickenAgent


## Step 4 — Run the Agent 🚀

Run a query against the agent. The agent will use its instructions to provide helpful responses.

In [13]:
# Run without approval mode - query the agent
query = "What are some best practices for managing rotisserie chicken inventory to minimize waste?"
print(f"🤔 User: {query}")
print("⏳ Waiting for response...\n")

result = await rotisserie_agent.run(query)
print(f"📊 {rotisserie_agent.name}: {result}")

🤔 User: What are some best practices for managing rotisserie chicken inventory to minimize waste?
⏳ Waiting for response...

📊 RotisserieChickenAgent: Here are some best practices for managing rotisserie chicken inventory to minimize waste:

### 1. Track Sales Patterns
- **Monitor hourly sales**—Identify your busiest times (often lunch 11am-1pm & dinner 5pm-7pm).
- **Review historical data**—Look for consistent trends on weekdays, weekends, and holidays.

### 2. Cook in Batches
- **Adjust batch sizes**—Cook smaller batches outside of peak hours; larger batches during busier periods.
- **Frequent, smaller cooking rounds**—Prevents overproduction and keeps chickens fresh for customers.

### 3. Use Sell-Through Rates
- **Calculate sell-through rate:** (Chickens sold ÷ chickens cooked) x 100.
- **Aim for high sell-through**—Rates of 85% or higher mean most chickens are sold, reducing waste.

### 4. Communication & Teamwork
- **Share daily targets with staff**—Ensure everyone knows goals fo

## Step 5 — Ask Follow-up Questions

Continue the conversation with follow-up queries:

In [14]:
# Another query to the agent
query = """What cooking schedule would you recommend for a typical weekday to maximize sales 
while minimizing waste? Consider typical peak hours."""

print(f"🤔 User: {query}")
print("⏳ Waiting for response...\n")

result = await rotisserie_agent.run(query)
print(f"📊 {rotisserie_agent.name}: {result}")

🤔 User: What cooking schedule would you recommend for a typical weekday to maximize sales 
while minimizing waste? Consider typical peak hours.
⏳ Waiting for response...

📊 RotisserieChickenAgent: Certainly! Here’s a recommended weekday rotisserie chicken cooking schedule based on typical sales patterns:

### Key Patterns
- **Lunch Rush:** 11am–1pm  
- **Dinner Rush:** 5pm–7pm  
- **Slower Hours:** Early morning, mid-afternoon, late evening

---

### Sample Cooking Schedule

#### **Morning (8am–11am):**
- **Cook small batches:** Start with a low quantity (e.g., 6–10 chickens). Most morning sales are low, mainly targeting early shoppers.

#### **Before Lunch Peak (11am):**
- **Increase quantity:** Prepare a larger batch to be ready at 11am (e.g., 12–18 chickens).  
- **Monitor sales:** Cook additional chickens in small batches if sales are strong.

#### **After Lunch (1pm–4pm):**
- **Cook to demand:** Reduce batch size back to 6–10 chickens.  
- **Check inventory:** Cook only if you’re 

## Step 6 — Adding Tools with Approval Mode 🔐

You can add function tools to the agent using the `@tool` decorator. The `approval_mode` parameter controls whether user confirmation is required.

```python
from typing import Annotated
from pydantic import Field

@tool(approval_mode="never_require")  # or "always_require" for production
def my_tool(
    param: Annotated[str, Field(description="Parameter description")]
) -> str:
    """Tool description shown to the model."""
    return "result"

agent = Agent(
    client=client,
    name="MyAgent",
    instructions="...",
    tools=[my_tool],
)
```

> ⚠️ **Note:** Use `approval_mode="always_require"` for write operations in production.

In [15]:
from typing import Annotated
from pydantic import Field

# Example: Define a tool with approval mode
@tool(approval_mode="never_require")
def get_store_recommendation(
    store_id: Annotated[str, Field(description="The store ID (e.g., Store-001)")],
    date: Annotated[str, Field(description="The date in YYYY-MM-DD format")],
) -> str:
    """Get cooking recommendations for a specific store and date."""
    # In a real scenario, this would query the MCP server
    return f"Recommendation for {store_id} on {date}: Cook 50 chickens at 10am, 30 at 2pm, and 40 at 5pm based on historical patterns."

# Create agent with the tool
agent_with_tools = Agent(
    client=client,
    name="RotisserieManagerAgent",
    instructions=ROTISSERIE_AGENT_INSTRUCTIONS,
    tools=[get_store_recommendation],
)

print(f"✅ Created agent with tools: {agent_with_tools.name}")

✅ Created agent with tools: RotisserieManagerAgent


## Step 7 — Test Agent with Tools

Run the agent with tools to see the tool invocation in action:

In [16]:
# Test the agent with tools
query = "Get cooking recommendations for Store-001 for 2024-02-01"
print(f"🤔 User: {query}")
print("⏳ Waiting for response...\n")

result = await agent_with_tools.run(query)
print(f"📊 {agent_with_tools.name}: {result}")

🤔 User: Get cooking recommendations for Store-001 for 2024-02-01
⏳ Waiting for response...

📊 RotisserieManagerAgent: Here are the cooking recommendations for Store-001 for February 1, 2024:

- Cook 50 chickens at 10am (to prepare for the lunch rush)
- Cook 30 chickens at 2pm (to cover steady afternoon sales)
- Cook 40 chickens at 5pm (for the dinner rush)

These numbers are based on historical sales patterns and are designed to match demand during peak hours while minimizing waste. If you notice sales trends changing (like higher or lower traffic), consider adjusting these numbers for future days.


## Key Takeaways 📚

### Creating an Agent with FoundryChatClient

```python
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

# Create the chat client
client = FoundryChatClient(
    project_endpoint="https://your-project.services.ai.azure.com",
    model="gpt-4o",
    credential=AzureCliCredential(),
)

# Create the agent
agent = Agent(
    client=client,
    name="MyAgent",
    instructions="You are a helpful assistant.",
)

# Run the agent
result = await agent.run("Hello!")
```

### Adding Function Tools

```python
from typing import Annotated
from pydantic import Field

@tool(approval_mode="never_require")
def my_function(
    param: Annotated[str, Field(description="Parameter description")]
) -> str:
    """Function description for the model."""
    return "result"

agent = Agent(
    client=client,
    name="AgentWithTools",
    instructions="...",
    tools=[my_function],
)
```

### Approval Modes

| Mode | Description | Recommended For |
|------|-------------|------------------|
| `"never_require"` | Tool calls execute automatically | Read-only queries, development |
| `"always_require"` | Every tool call requires explicit approval | Write operations, production |

### Best Practices

1. **Security First**: Use `approval_mode="always_require"` for write operations
2. **Clear Instructions**: Provide detailed system instructions for the agent
3. **Tool Descriptions**: Use clear docstrings and Field descriptions for tools
4. **Error Handling**: Handle network failures gracefully
5. **Agent Names**: Use PascalCase for agent names (e.g., `RotisserieChickenAgent`)

### Architecture Overview

```
┌──────────────────┐     ┌─────────────────────┐     ┌──────────────────────┐
│  Your Notebook   │────▶│  Azure AI Foundry   │────▶│  Deployed Model      │
│  (Agent + Client)│     │  (Project Endpoint) │     │  (gpt-4o, etc.)      │
└──────────────────┘     └─────────────────────┘     └──────────────────────┘
        │                         
        │                         
   AzureCliCredential             
   Authentication                 
```

⚠️ **Security Note**: For production deployments:
- Use `approval_mode="always_require"` for sensitive operations
- Implement proper authentication
- Log all tool invocations for audit purposes

## Troubleshooting

| Issue | Solution |
|-------|----------|
| `AI_FOUNDRY_PROJECT_ENDPOINT not set` | Create a `.env` file with your Foundry project endpoint |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME not set` | Add the model deployment name to your `.env` file |
| `Authentication failed` | Run `az login --use-device-code` in your terminal |
| `Model not found` | Verify the model deployment exists in your Foundry project |
| `Import error` | Run `%pip install agent-framework-core==1.0.1` to install dependencies |
| `Tool not called` | Check that tool descriptions and parameter annotations are clear |